# Image-Level EDA
The bias matching process applied in the previous metadata EDA notebook tells us that the bias matching can be applied to reduce the bias in the dataset. However, it is also important to perform an image-level EDA to understand the distribution of images and their characteristics.

Crucially, this notebook combines multiple per source manifests in `data/interim` (produced by `scripts/build_manifest.py`) into a single combined manifest (`data/interim/manifest_final.parquet`) with paths pointing to the final, model-ready image folders under `data/processed/`, allowing for an overall image analysis. Train/val/test splits are allocated here.

The pipeline this notebook follows is as follows:


```text
per-source manifests (data/interim/*.parquet)
        |
        v
combine_manifests()          -- into one DataFrame standardized to canonical columns
        |
        v
drop is_corrupt rows          -- ignored going forward
        |
        v
assign_group_ids()             -- cross-source near-duplicate clustering (banded LSH)
        |
        v
apply_matching()                -- bias-match real GenImage images (size + JPEG QF)
        |
        v
assign_full_splits()           -- group-aware GenImage split + fixed lookup for
                                   coco / ntire / raise
        |
        v
assert_no_leakage()             -- hard stop if any group/hash spans two splits
        |
        v
save data/interim/manifest_final.parquet
        |
        v
build_processed_dataset()      -- resize + re-encode every surviving image
        |
        v
data/processed/<split>/<ai|human>/<sha256>.jpg
        +
data/processed/manifest.parquet

### 0. Setup

In [2]:
%load_ext autoreload
%autoreload 2

## Standard Libraries
from pathlib import Path
import numpy as np
import pandas as pd
import yaml

## Project module imports
from ai_detector.data.selection import(
    SubsetConfig, combine_manifests, assign_group_ids, apply_matching, assign_full_splits, 
    normalize_columns, shortcut_probe)
from ai_detector.data.manifest import(assert_no_leakage, LeakageError)
from ai_detector.data import viz
from ai_detector.preprocessing.image_ops import PreprocessConfig, build_processed_dataset

## Path setup
PROJECT_ROOT = Path.cwd().parent
DATA = PROJECT_ROOT / "data"
INTERIM = DATA / "interim"
PROCESSED = DATA / "processed"
RAW = DATA / "raw"
CONFIG_PATH = PROJECT_ROOT / "configs" / "data" / "subset_v1.yaml"

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)

## Preprocessing and data splitting will draw from a yaml file in configs/data
cfg = SubsetConfig.from_yaml(CONFIG_PATH)
raw_cfg = yaml.safe_load(CONFIG_PATH.read_text())
preprocess_cfg = PreprocessConfig.from_yaml_dict(raw_cfg.get("preprocessing", {}))

print(cfg)
print(preprocess_cfg)

SubsetConfig(name='subset_v1', seed=42, real_generator_token='nature', train_generators=['stable_diffusion_v_1_4', 'stable_diffusion_v_1_5', 'glide', 'adm', 'vqdm'], ood_generators=['midjourney', 'wukong', 'biggan'], min_side=450, max_side=550, jpeg_qf=98, jpeg_qf_tolerance=2, val_fraction=0.1, test_in_dist_fraction=0.1, pilot_n_per_stratum=60)
PreprocessConfig(image_size=512, jpeg_qf=98, resample=<Resampling.LANCZOS: 1>)


### 1. Load and combine per-source manifests
Draws from the per-source manifests stored in `data/interim`

In [5]:
manifest_paths = sorted(INTERIM.glob("manifest_*.parquet"))
print(f"Found {len(manifest_paths)} manifest files:")
for p in manifest_paths:
    print(" -", p.name)

df = combine_manifests(manifest_paths)
print(f"\nCombined manifest: {len(df):,} rows, {df['source'].nunique()} sources")
display(df.groupby(["source", "label"]).size().unstack(fill_value=0))

Found 4 manifest files:
 - manifest_coco.parquet
 - manifest_genimage_tiny.parquet
 - manifest_ntire.parquet
 - manifest_raise.parquet

Combined manifest: 90,300 rows, 4 sources


label,0,1
source,,
coco,5000,0
genimage,17500,17500
ntire,17982,32018
raise,300,0


The combined manifests is sampled below:

In [9]:
display(df.sample(7))
print(df["is_corrupt"].value_counts())

,path,source,sha256,phash,label,generator,split,content_class,group_id,width,height,file_format,file_size_bytes,jpeg_qf,mode,is_corrupt,error
56092,ntire/shard_0/images/525b16ce8f4746521c2a.jpg,ntire,b98d71b82f2fbe197b53b2ae6e6703eb06919aab4f9be0...,390b9668c3b4cce3,0,mixed,test_wild,None,None,2240,3328,JPEG,2004168,95,RGB,False,None
72614,ntire/shard_0/images/a65ffab67f57db8210b4.jpg,ntire,a6c3491d68184b42cd601b09f01938d971f3b51e75ee5f...,21d826c8d64ddb35,1,mixed,test_wild,None,None,384,384,JPEG,62603,95,RGB,False,None
59064,ntire/shard_0/images/616bbae696ae535e4119.jpg,ntire,b0ace29849f2a84fb55af6da9602f38688368781cc1392...,428bd135c6969da9,1,mixed,test_wild,None,None,2048,1536,JPEG,549950,95,RGB,False,None
32395,tiny_genimage/imagenet_glide/train/nature/n020...,genimage,311bafada14024e37a45f7d4e23714c6c0d3e220ed4e05...,4c3c3343666d994b,0,imagenet_glide,train,None,None,500,333,JPEG,185277,96.0,RGB,False,None
24054,tiny_genimage/imagenet_ai_0424_wukong/val/ai/1...,genimage,872536e8710761706ccefdc63e6c94fb602f2895e8e1d6...,5746a83d7d29d282,1,imagenet_ai_0424_wukong,val,None,None,512,512,PNG,408173,NaN,RGB,False,None
14990,tiny_genimage/imagenet_ai_0419_vqdm/val/nature...,genimage,d80fe742cb3aeb6b486d231f2cfaf934083b3d01f245fe...,55c3cb152c7cc683,0,imagenet_ai_0419_vqdm,val,None,None,500,375,JPEG,88702,96.0,RGB,False,None
36374,tiny_genimage/imagenet_midjourney/train/ai/720...,genimage,2593e41124afc27f48f24cbb346353b80e48f7b8729108...,6c9a9b65b1619632,1,imagenet_midjourney,train,None,None,1024,1024,PNG,578908,NaN,RGB,False,None


is_corrupt
False    90300
Name: count, dtype: int64


### 2. Drop corrupt image rows
In the event corrupt images are present. These are flagged by `probe_decodability()` when the individual source-specific manifests were created. They lack a usable `phash` and therefore cannot be preprocessed/grouped/split, therefore they must be removed.

In [7]:
n_before= len(df)
corrupt= df[df["is_corrupt"]]

if len(corrupt):
    display(corrupt[["path", "source", "error"]].head(10))
df = df[~df["is_corrupt"]].reset_index(drop=True)
print(f"Dropped {n_before - len(df)} corrupt rows ({(n_before - len(df)) / n_before:.2%})")
print(f"Remaining: {len(df):,} rows")

Dropped 0 corrupt rows (0.00%)
Remaining: 90,300 rows


### 3. Cross-source Near-duplicate clustering
We initially implemented a Banded LSH + UnionFind (`integrity.py`) pipeline to group similar images within a particular source. This time, we apply the same grouping strategy across the entire combined manifest. 